# 01 - GPU Cluster Simulator Exploration

This notebook demonstrates the discrete-event simulator, cluster state, queue transitions, and invariant checks under simulated workloads.

In [1]:
import sys
import os
import matplotlib.pyplot as plt

# Ensure project root is in python path
sys.path.insert(0, os.path.abspath('..'))

from simulator.cluster import Cluster
from simulator.job import Job, WorkloadType
from simulator.simulator import Simulator

print("Simulator imported successfully!")

Simulator imported successfully!


## 1. Load Heterogeneous Cluster Configuration

In [2]:
cluster = Cluster.from_yaml("../configs/cluster_medium.yaml")
print(f"Nodes count: {cluster.num_nodes}")
print(f"Total GPUs: {cluster.total_gpus}")
print(f"Total VRAM: {cluster.total_vram_gb} GB")

for node in cluster.nodes:
    print(f"Node {node.node_id}: {node.gpu_count}x {node.gpu_type} ({node.vram_per_gpu_gb}GB/GPU), Total VRAM: {node.total_vram_gb}GB")

Nodes count: 4
Total GPUs: 24
Total VRAM: 1152.0 GB
Node 0: 4x A100_80GB (80.0GB/GPU), Total VRAM: 320.0GB
Node 1: 8x A100_40GB (40.0GB/GPU), Total VRAM: 320.0GB
Node 2: 4x H100_80GB (80.0GB/GPU), Total VRAM: 320.0GB
Node 3: 8x A10_24GB (24.0GB/GPU), Total VRAM: 192.0GB


## 2. Simulate Workload with Discrete Events

In [ ]:
sim = Simulator(cluster=cluster, max_queue_size=16, horizon_seconds=1800.0)
sim.reset()

# Submit sample jobs
sample_jobs = [
    Job(job_id=1, arrival_time=0.0, gpu_count=4, vram_per_gpu_gb=80.0, estimated_runtime=300.0, actual_runtime=300.0, workload_type=WorkloadType.TRAINING),
    Job(job_id=2, arrival_time=20.0, gpu_count=2, vram_per_gpu_gb=40.0, estimated_runtime=150.0, actual_runtime=150.0, workload_type=WorkloadType.FINE_TUNING),
    Job(job_id=3, arrival_time=50.0, gpu_count=8, vram_per_gpu_gb=24.0, estimated_runtime=200.0, actual_runtime=200.0, workload_type=WorkloadType.EXPERIMENT),
    Job(job_id=4, arrival_time=80.0, gpu_count=1, vram_per_gpu_gb=20.0, estimated_runtime=60.0, actual_runtime=60.0, workload_type=WorkloadType.INFERENCE),
]
sim.load_workload(sample_jobs)

print(f"Submitted {len(sample_jobs)} jobs. Running simulation...")

done = False
while not done:
    done, completed = sim.step_to_next_decision()
    if done:
        break
    state = sim.get_state()
    mask = state.get_action_mask()
    
    # Greedy assignment
    for j_idx in range(len(state.queue)):
        scheduled = False
        for n_idx in range(state.num_nodes):
            if mask[j_idx, n_idx] > 0:
                sim.apply_action(j_idx, n_idx)
                scheduled = True
                break
        if scheduled:
            break

metrics = sim.get_metrics()
print("Simulation completed!")
for k, v in metrics.items():
    print(f"{k}: {v}")

Submitted 4 jobs. Running simulation...
Simulation completed!
completed_jobs: 4
submitted_jobs: 4
mean_jct: 177.5
p95_jct: 284.99999999999994
mean_wait_time: 0.0
gpu_utilization: 0.07314814814814814
throughput_jobs_per_hour: 8.0
deadline_violation_rate: 0.0
invalid_action_count: 0
simulation_duration: 1800.0


: 